In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
import time
import pandas as pd
import re
import json

BASE = "https://www.larazon.es"

# Puedes ampliar o reducir páginas aquí
TAG_URLS = ["https://www.larazon.es/tags/estrecho-de-ormuz"]
for i in range(2, 11):
    TAG_URLS.append(f"https://www.larazon.es/tags/estrecho-de-ormuz-{i}")

START_DATE = datetime(2026, 1, 1)
END_DATE = datetime(2026, 4, 21, 23, 59, 59)

# -----------------------------
# CONFIGURACIÓN SELENIUM
# -----------------------------
options = Options()
options.headless = False   # pon True si no quieres ver el navegador
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# Para no aceptar cookies en cada página
cookies_aceptadas = False


def get_soup_selenium(url, wait_seconds=3):
    print(f"\nAbriendo: {url}")
    driver.get(url)
    time.sleep(wait_seconds)
    return BeautifulSoup(driver.page_source, "html.parser")


def aceptar_cookies_si_aparece():
    global cookies_aceptadas

    if cookies_aceptadas:
        return

    try:
        botones = driver.find_elements(By.TAG_NAME, "button")
        for boton in botones:
            texto = boton.text.strip().lower()
            if any(x in texto for x in ["acept", "consent", "agree", "aceptar"]):
                print("Aceptando cookies...")
                boton.click()
                time.sleep(2)
                cookies_aceptadas = True
                return
    except Exception as e:
        print(f"No se pudieron gestionar cookies: {e}")


def limpiar_texto(texto):
    if not texto:
        return None
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto if texto else None


def extraer_links_tag(url):
    soup = get_soup_selenium(url, wait_seconds=4)

    aceptar_cookies_si_aparece()
    soup = BeautifulSoup(driver.page_source, "html.parser")

    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        texto_link = a.get_text(" ", strip=True).lower()

        if href.startswith("/"):
            full = urljoin(BASE, href)
        else:
            full = href

        full_lower = full.lower()

        # Nos quedamos solo con artículos HTML de La Razón
        # y relacionados con Ormuz por URL o por texto del enlace
        if (
            "larazon.es" in full_lower
            and full_lower.endswith(".html")
            and (
                "ormuz" in full_lower
                or "estrecho-de-ormuz" in full_lower
                or "ormuz" in texto_link
                or "estrecho de ormuz" in texto_link
            )
        ):
            links.add(full)

    print(f"Enlaces encontrados en la tag: {len(links)}")
    return sorted(links)


def extraer_fecha(soup):
    fecha = None

    # Intento 1: meta article:published_time
    meta = soup.find("meta", attrs={"property": "article:published_time"})
    if meta and meta.get("content"):
        try:
            fecha = datetime.fromisoformat(
                meta["content"].replace("Z", "+00:00")
            ).replace(tzinfo=None)
            return fecha
        except Exception:
            pass

    # Intento 2: JSON-LD
    scripts = soup.find_all("script", type="application/ld+json")
    for s in scripts:
        try:
            contenido = s.string if s.string else s.get_text(" ", strip=True)
            if not contenido:
                continue

            # Buscar datePublished con regex
            m = re.search(r'"datePublished"\s*:\s*"([^"]+)"', contenido)
            if m:
                try:
                    fecha = datetime.fromisoformat(
                        m.group(1).replace("Z", "+00:00")
                    ).replace(tzinfo=None)
                    return fecha
                except Exception:
                    pass

            # Intentar parsear como JSON
            try:
                data = json.loads(contenido)

                def buscar_fecha(obj):
                    if isinstance(obj, dict):
                        if "datePublished" in obj:
                            return obj["datePublished"]
                        for v in obj.values():
                            res = buscar_fecha(v)
                            if res:
                                return res
                    elif isinstance(obj, list):
                        for item in obj:
                            res = buscar_fecha(item)
                            if res:
                                return res
                    return None

                fecha_str = buscar_fecha(data)
                if fecha_str:
                    try:
                        fecha = datetime.fromisoformat(
                            fecha_str.replace("Z", "+00:00")
                        ).replace(tzinfo=None)
                        return fecha
                    except Exception:
                        pass
            except Exception:
                pass

        except Exception:
            pass

    # Intento 3: texto tipo Creada:
    texto = soup.get_text(" ", strip=True)
    m = re.search(r"Creada:\s*(\d{2}\.\d{2}\.\d{4})", texto)
    if m:
        try:
            return datetime.strptime(m.group(1), "%d.%m.%Y")
        except Exception:
            pass

    # Intento 4: otras fechas visibles
    patrones = [
        (r'(\d{4}-\d{2}-\d{2})', "%Y-%m-%d"),
        (r'(\d{2}/\d{2}/\d{4})', "%d/%m/%Y"),
        (r'(\d{2}\.\d{2}\.\d{4})', "%d.%m.%Y"),
    ]

    for patron, fmt in patrones:
        m = re.search(patron, texto)
        if m:
            try:
                return datetime.strptime(m.group(1), fmt)
            except Exception:
                pass

    return None


def extraer_titulo(soup):
    # Mejor primero og:title
    meta_title = soup.find("meta", attrs={"property": "og:title"})
    if meta_title and meta_title.get("content"):
        return limpiar_texto(meta_title["content"])

    if soup.title:
        return limpiar_texto(soup.title.get_text(" ", strip=True))

    h1 = soup.find("h1")
    if h1:
        return limpiar_texto(h1.get_text(" ", strip=True))

    return None


def extraer_texto_noticia(soup):
    # Intento 1: article
    article = soup.find("article")
    if article:
        parrafos = article.find_all("p")
        textos = []
        for p in parrafos:
            t = limpiar_texto(p.get_text(" ", strip=True))
            if t and len(t) > 30:
                textos.append(t)

        if textos:
            return " ".join(textos)

    # Intento 2: main
    main = soup.find("main")
    if main:
        parrafos = main.find_all("p")
        textos = []
        for p in parrafos:
            t = limpiar_texto(p.get_text(" ", strip=True))
            if t and len(t) > 30:
                textos.append(t)

        if textos:
            return " ".join(textos)

    # Intento 3: todos los párrafos
    parrafos = soup.find_all("p")
    textos = []
    for p in parrafos:
        t = limpiar_texto(p.get_text(" ", strip=True))
        if t and len(t) > 30:
            textos.append(t)

    # Quitamos duplicados manteniendo orden
    vistos = set()
    textos_unicos = []
    for t in textos:
        if t not in vistos:
            vistos.add(t)
            textos_unicos.append(t)

    if textos_unicos:
        return " ".join(textos_unicos)

    return None


def extraer_articulo(url):
    soup = get_soup_selenium(url, wait_seconds=3)

    titulo = extraer_titulo(soup)
    fecha = extraer_fecha(soup)
    texto = extraer_texto_noticia(soup)

    return {
        "periodico": "La Razón",
        "fecha": fecha,
        "titulo": titulo,
        "texto": texto,
        "url": url
    }


# -----------------------------
# RECOGER LINKS
# -----------------------------
todos_links = set()

for tag_url in TAG_URLS:
    try:
        print(f"\nProcesando página de tag: {tag_url}")
        links = extraer_links_tag(tag_url)
        todos_links.update(links)
        print(f"Total acumulado de links únicos: {len(todos_links)}")
        time.sleep(2)
    except Exception as e:
        print(f"Error en {tag_url}: {e}")

# -----------------------------
# PROCESAR ARTÍCULOS
# -----------------------------
resultados = []

for i, link in enumerate(sorted(todos_links), start=1):
    try:
        print(f"\n[{i}/{len(todos_links)}] Procesando artículo:")
        print(link)

        data = extraer_articulo(link)

        print(f"Título: {data['titulo']}")
        print(f"Fecha: {data['fecha']}")
        print(f"Longitud texto: {len(data['texto']) if data['texto'] else 0}")

        if data["fecha"] is None:
            print("-> Sin fecha, se descarta")
        elif START_DATE <= data["fecha"] <= END_DATE:
            print("-> Dentro del rango, se añade")
            resultados.append(data)
        else:
            print("-> Fuera del rango")

        time.sleep(2)

    except Exception as e:
        print(f"Error en artículo {link}: {e}")

driver.quit()

# -----------------------------
# RESULTADOS
# -----------------------------
if resultados:
    df = pd.DataFrame(resultados)

    # Quitar duplicados por URL
    df = df.drop_duplicates(subset=["url"])

    # Ordenar por fecha
    df = df.sort_values("fecha")

    # Orden exacto de columnas
    df = df[["periodico", "fecha", "titulo", "texto", "url"]]

    print("\nRESULTADOS FINALES:")
    print(df[["periodico", "fecha", "titulo", "url"]])
    print("\nTOTAL NOTICIAS:", len(df))

    df.to_csv("data_larazon.csv", index=False, encoding="utf-8-sig")
    print("\nArchivo guardado como: data_larazon.csv")
else:
    print("\nNo se encontraron resultados en el rango indicado.")


Procesando página de tag: https://www.larazon.es/tags/estrecho-de-ormuz

Abriendo: https://www.larazon.es/tags/estrecho-de-ormuz
Aceptando cookies...
Enlaces encontrados en la tag: 25
Total acumulado de links únicos: 25

Procesando página de tag: https://www.larazon.es/tags/estrecho-de-ormuz-2

Abriendo: https://www.larazon.es/tags/estrecho-de-ormuz-2
Enlaces encontrados en la tag: 23
Total acumulado de links únicos: 48

Procesando página de tag: https://www.larazon.es/tags/estrecho-de-ormuz-3

Abriendo: https://www.larazon.es/tags/estrecho-de-ormuz-3
Enlaces encontrados en la tag: 18
Total acumulado de links únicos: 66

Procesando página de tag: https://www.larazon.es/tags/estrecho-de-ormuz-4

Abriendo: https://www.larazon.es/tags/estrecho-de-ormuz-4
Enlaces encontrados en la tag: 17
Total acumulado de links únicos: 83

Procesando página de tag: https://www.larazon.es/tags/estrecho-de-ormuz-5

Abriendo: https://www.larazon.es/tags/estrecho-de-ormuz-5
Enlaces encontrados en la tag: 8


In [ ]:
import os

ruta = os.path.abspath("data_larazon.csv")
df.to_csv(ruta, index=False, encoding="utf-8-sig")

print("Archivo guardado en:", ruta)

Archivo guardado en: c:\Users\iyana\AppData\Local\Programs\Microsoft VS Code\noticias_ormuz_larazon.csv
